<a href="https://colab.research.google.com/github/kennethdave005-star/INDEXING-TOOLS/blob/main/INDEX_CALCULATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
National Campus Cost of Living Index (NCCLI) - Interactive Student Budget & Index Calculator
=============================================================================================
Framework: 8-Dimension Student Expenditure Model (Excluding Healthcare)
Formula:   NCCLI = SUM(W_i * N_i)
  where:
    W_i = Category Weight (summing to 1.0 or 100%)
    N_i = Sub-Index (Student Expense / Baseline Benchmark * 100)

Usage:
  - Run demo mode:         python3 nccli_budget_calculator.py --demo
  - Interactive mode:      python3 nccli_budget_calculator.py --interactive
  - Command-line mode:     python3 nccli_budget_calculator.py --housing 35000 --food 25000 --transport 15000 ...
"""

import sys
import argparse
import json
from typing import Dict, Any

# 8 Core Categories and Default Relative Weights (Normalized to 100%)
# Base weights derived from the 8-Dimension NCCLI Framework:
# Housing: 26.6%, Food: 23.4%, Transport: 14.9%, Digital: 10.6%, Academic: 9.6%, Energy: 5.3%, Personal: 5.3%, Other: 4.3%
DEFAULT_CATEGORIES = {
    "housing": {
        "name": "Accommodation & Housing",
        "weight": 0.266,
        "baseline_ngn": 30000.0,
        "description": "Rent, hostel fees, tenancy deposits, utilities"
    },
    "food": {
        "name": "Food & Nutrition",
        "weight": 0.234,
        "baseline_ngn": 25000.0,
        "description": "Daily meals, groceries, cooking fuel, drinking water"
    },
    "transport": {
        "name": "Transportation & Mobility",
        "weight": 0.149,
        "baseline_ngn": 15000.0,
        "description": "Daily campus shuttles, local transit, travel home"
    },
    "digital": {
        "name": "Digital Connectivity",
        "weight": 0.106,
        "baseline_ngn": 10000.0,
        "description": "Mobile data, Wi-Fi subscriptions, online learning"
    },
    "academic": {
        "name": "Academic & Educational",
        "weight": 0.096,
        "baseline_ngn": 9000.0,
        "description": "Textbooks, printing, handouts, lab supplies, software"
    },
    "energy": {
        "name": "Energy & Electricity",
        "weight": 0.053,
        "baseline_ngn": 5000.0,
        "description": "Generator fuel, solar charging, electricity tariffs"
    },
    "personal": {
        "name": "Personal & Lifestyle",
        "weight": 0.053,
        "baseline_ngn": 5000.0,
        "description": "Upkeep, laundry services, basic living miscellany"
    },
    "other": {
        "name": "Other Financial Obligations",
        "weight": 0.043,
        "baseline_ngn": 4000.0,
        "description": "Bank charges, unexpected emergency outlays, fees"
    }
}


class NCCLIBudgetCalculator:
    def __init__(self, categories: Dict[str, Any] = None):
        self.categories = categories or DEFAULT_CATEGORIES

    def calculate_index(self, student_expenses: Dict[str, float]) -> Dict[str, Any]:
        """
        Calculate the composite NCCLI index score and category breakdown.
        """
        total_expense = 0.0
        total_baseline = 0.0
        composite_nccli = 0.0
        breakdown = {}

        for cat_key, cat_info in self.categories.items():
            expense = float(student_expenses.get(cat_key, 0.0))
            baseline = cat_info["baseline_ngn"]
            weight = cat_info["weight"]

            # Sub-index N_i: (Student Expense / Baseline) * 100
            sub_index = (expense / baseline * 100.0) if baseline > 0 else 0.0

            # Weighted contribution: W_i * N_i
            weighted_score = weight * sub_index

            total_expense += expense
            total_baseline += baseline
            composite_nccli += weighted_score

            breakdown[cat_key] = {
                "name": cat_info["name"],
                "expense_ngn": expense,
                "baseline_ngn": baseline,
                "weight_pct": weight * 100,
                "sub_index": sub_index,
                "weighted_contribution": weighted_score,
                "variance_pct": ((expense - baseline) / baseline * 100) if baseline > 0 else 0.0
            }

        # Determine financial risk profile
        risk_profile = self._assess_financial_risk(composite_nccli, total_expense, student_expenses)

        return {
            "total_monthly_expense_ngn": total_expense,
            "national_baseline_expense_ngn": total_baseline,
            "composite_nccli_score": round(composite_nccli, 2),
            "risk_profile": risk_profile,
            "category_breakdown": breakdown
        }

    def _assess_financial_risk(self, nccli_score: float, total_expense: float, expenses: Dict[str, float]) -> str:
        if nccli_score > 130:
            status = "HIGH FINANCIAL VULNERABILITY (Expenses 30%+ above baseline)"
        elif nccli_score > 110:
            status = "MODERATE FINANCIAL PRESSURE (Expenses 10-30% above baseline)"
        elif nccli_score >= 90:
            status = "BALANCED STUDENT BUDGET (Within 10% of national baseline)"
        else:
            status = "LOW EXPENDITURE / CONSTRAINED BUDGET (Below national baseline)"

        # Check category-specific vulnerabilities
        housing_ratio = expenses.get("housing", 0) / total_expense if total_expense > 0 else 0
        if housing_ratio > 0.40:
            status += " | Warning: High Housing Rent Burden (>40% of total budget)"

        return status

    def print_dashboard(self, results: Dict[str, Any], student_name: str = "Student"):
        """Print formatted visual dashboard to terminal."""
        print("\n" + "=" * 80)
        print(f"  NATIONAL CAMPUS COST OF LIVING INDEX (NCCLI) - STUDENT REPORT")
        print(f"  Profile: {student_name}")
        print("=" * 80)

        print(f"\n  ► Total Monthly Expenditure:  NGN {results['total_monthly_expense_ngn']:,.2f}")
        print(f"  ► National Baseline Expense:   NGN {results['national_baseline_expense_ngn']:,.2f}")
        print(f"  ► COMPOSITE NCCLI SCORE:       {results['composite_nccli_score']} (Baseline = 100.00)")
        print(f"  ► Vulnerability Assessment:   {results['risk_profile']}\n")

        print("-" * 80)
        print(f"{'Category Name':<30} | {'Expense (NGN)':<14} | {'Baseline':<12} | {'Sub-Index':<10} | {'Contribution'}")
        print("-" * 80)

        for key, item in results["category_breakdown"].items():
            print(f"{item['name']:<30} | ₦{item['expense_ngn']:>12,.2f} | ₦{item['baseline_ngn']:>10,.2f} | {item['sub_index']:>9.1f}  | {item['weighted_contribution']:>8.2f}")

        print("-" * 80)
        print(f"Formula: NCCLI = Σ(W_i × N_i) = {results['composite_nccli_score']:.2f}")
        print("=" * 80 + "\n")


def run_interactive(calculator: NCCLIBudgetCalculator):
    """Interactive CLI mode for terminal users."""
    print("\n--- NCCLI Interactive Student Budget Calculator ---")
    print("Enter your monthly expenditure in Nigerian Naira (NGN) for each category:\n")

    expenses = {}
    for key, info in calculator.categories.items():
        while True:
            try:
                user_input = input(f"  [{info['name']}] (Baseline: ₦{info['baseline_ngn']:,.0f}): ₦")
                if not user_input.strip():
                    expenses[key] = info['baseline_ngn']
                else:
                    expenses[key] = float(user_input)
                break
            except ValueError:
                print("    Invalid input. Please enter a valid numerical value.")

    name = input("\nEnter Student / Campus Name (Optional): ").strip() or "Anonymous Student"
    results = calculator.calculate_index(expenses)
    calculator.print_dashboard(results, student_name=name)


def main():
    parser = argparse.ArgumentParser(description="NCCLI Student Budget & Index Calculator")
    parser.add_argument("--demo", action="store_true", help="Run automated demonstration with sample student data")
    parser.add_argument("--interactive", action="store_true", help="Launch interactive CLI calculator")
    parser.add_argument("--export", type=str, help="Save report to specified JSON file path")

    # Category arguments for headless execution
    for key, info in DEFAULT_CATEGORIES.items():
        parser.add_argument(f"--{key}", type=float, default=None, help=f"Monthly expenditure for {info['name']}")

    # Use parse_known_args to ignore unknown arguments passed by the Colab kernel
    args, unknown = parser.parse_known_args()
    calculator = NCCLIBudgetCalculator()

    if args.demo or (len(sys.argv) == 1 and not sys.stdin.isatty()):
        print("\n[Running Demo Mode with Sample Student Expenditure Data]")
        sample_expenses = {
            "housing": 38000.0,  # Higher off-campus rent
            "food": 30000.0,     # Food inflation impact
            "transport": 18000.0,# Shuttle & commute
            "digital": 12000.0,  # Data plans
            "academic": 10000.0, # Books & printing
            "energy": 7000.0,    # Generator fuel contribution
            "personal": 6000.0,   # Laundry & upkeep
            "other": 5000.0      # Misc charges
        }
        results = calculator.calculate_index(sample_expenses)
        calculator.print_dashboard(results, student_name="Sample Student (Zone 4 Campus)")

        if args.export:
            with open(args.export, "w") as f:
                json.dump(results, f, indent=2)
            print(f"Report exported to {args.export}")
        return

    if args.interactive:
        run_interactive(calculator)
        return

    # Check if explicit category args were passed
    passed_expenses = {}
    any_arg = False
    for key in DEFAULT_CATEGORIES.keys():
        val = getattr(args, key, None)
        if val is not None:
            passed_expenses[key] = val
            any_arg = True
        else:
            passed_expenses[key] = DEFAULT_CATEGORIES[key]["baseline_ngn"]

    if any_arg:
        results = calculator.calculate_index(passed_expenses)
        calculator.print_dashboard(results, student_name="Custom CLI Entry")
        if args.export:
            with open(args.export, "w") as f:
                json.dump(results, f, indent=2)
            print(f"Report exported to {args.export}")
    else:
        # Fallback to interactive mode if TTY available, else demo
        if sys.stdin.isatty():
            run_interactive(calculator)
        else:
            print("[Non-interactive shell detected. Executing demo mode.]")
            results = calculator.calculate_index({k: v["baseline_ngn"] * 1.15 for k, v in DEFAULT_CATEGORIES.items()})
            calculator.print_dashboard(results, student_name="Automated Test Run")


if __name__ == "__main__":
    main()


[Non-interactive shell detected. Executing demo mode.]

  NATIONAL CAMPUS COST OF LIVING INDEX (NCCLI) - STUDENT REPORT
  Profile: Automated Test Run

  ► Total Monthly Expenditure:  NGN 118,450.00
  ► National Baseline Expense:   NGN 103,000.00
  ► COMPOSITE NCCLI SCORE:       115.0 (Baseline = 100.00)
  ► Vulnerability Assessment:   MODERATE FINANCIAL PRESSURE (Expenses 10-30% above baseline)

--------------------------------------------------------------------------------
Category Name                  | Expense (NGN)  | Baseline     | Sub-Index  | Contribution
--------------------------------------------------------------------------------
Accommodation & Housing        | ₦   34,500.00 | ₦ 30,000.00 |     115.0  |    30.59
Food & Nutrition               | ₦   28,750.00 | ₦ 25,000.00 |     115.0  |    26.91
Transportation & Mobility      | ₦   17,250.00 | ₦ 15,000.00 |     115.0  |    17.13
Digital Connectivity           | ₦   11,500.00 | ₦ 10,000.00 |     115.0  |    12.19
Academic

In [ ]:
# Initialize the calculator
calculator = NCCLIBudgetCalculator()

# Run the interactive mode
run_interactive(calculator)


--- NCCLI Interactive Student Budget Calculator ---
Enter your monthly expenditure in Nigerian Naira (NGN) for each category:

  [Accommodation & Housing] (Baseline: ₦30,000): ₦30000
  [Food & Nutrition] (Baseline: ₦25,000): ₦12000
  [Transportation & Mobility] (Baseline: ₦15,000): ₦25000
  [Digital Connectivity] (Baseline: ₦10,000): ₦30000
  [Academic & Educational] (Baseline: ₦9,000): ₦1000
  [Energy & Electricity] (Baseline: ₦5,000): ₦3500
  [Personal & Lifestyle] (Baseline: ₦5,000): ₦4000
  [Other Financial Obligations] (Baseline: ₦4,000): ₦2500

Enter Student / Campus Name (Optional): KENNETH

  NATIONAL CAMPUS COST OF LIVING INDEX (NCCLI) - STUDENT REPORT
  Profile: KENNETH

  ► Total Monthly Expenditure:  NGN 108,000.00
  ► National Baseline Expense:   NGN 103,000.00
  ► COMPOSITE NCCLI SCORE:       106.17 (Baseline = 100.00)
  ► Vulnerability Assessment:   BALANCED STUDENT BUDGET (Within 10% of national baseline)

---------------------------------------------------------------